In [1]:
! git clone https://github.com/westedcrean/master-thesis && pip install wandb loguru

Cloning into 'master-thesis'...
remote: Enumerating objects: 1673, done.
remote: Counting objects: 100% (335/335), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 1673 (delta 121), reused 292 (delta 98), pack-reused 1338
Receiving objects: 100% (1673/1673), 432.49 MiB | 16.37 MiB/s, done.
Resolving deltas: 100% (756/756), done.
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.9 MB 31.3 MB/s 
     |████████████████████████████████| 58 kB 6.5 MB/s 
     |████████████████████████████████| 168 kB 71.3 MB/s 
     |████████████████████████████████| 182 kB 59.4 MB/s 
     |████████████████████████████████| 63 kB 1.5 MB/s 
     |████████████████████████████████| 166 kB 67.8 MB/s 
     |████████████████████████████████| 166 kB 71.6 MB/s 
     |████████████████████████████████| 162 kB 67.8 MB/s 
     |████████████████████████████████| 162 kB 67.0 MB/s 
     |████████████████████████

In [2]:
! cd master-thesis && git pull

Already up to date.


In [3]:
! cp -r master-thesis/* .

In [4]:
! cd src && ./download_dataset.sh && python create_datasets.py

Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
Extracting  znaki/png/9/9_1358_98_M_2B.png                                99%  OK 
Extracting  znaki/png/9/9_1359_97_M_2F.png                                99%  OK 
Extracting  znaki/png/9/9_1359_98_M_2B.png                                99%  OK 
Extracting  znaki/png/9/9_1360_97_M_2F.png                                99%  OK 
Extracting  znaki/png/9/9_1360_98_M_2B.png                                99%  OK 
Extracting  znaki/png/9/9_1361_96_M_2B.png                                99%  OK 
Extracting  znaki/png/9/9_1361_97_M_2F.png                                99%  OK 
Extracting  znaki/png/9/9_1362_98_M_2D.png                                99%  OK 
Extracting  znaki/png/9/9_1362_98_M_2F.png                                99%  OK 
Extracting  znaki/png/9/9_1363_98_M_2F.png                                99%  OK 
Extracting  zna

In [5]:
from pathlib import Path

import wandb
from wandb.keras import WandbCallback

import matplotlib.pyplot as plt
import tensorflow as tf
import os
import sys

sys.path.append(str((Path.cwd().absolute() / "src").resolve()))

In [6]:
from pathlib import Path

import wandb
from wandb.keras import WandbCallback

import matplotlib.pyplot as plt
import tensorflow as tf
from loguru import logger


from training.engine import train, test
from training.create_models import get_models_for_experiment
from datasets import (
    lowercase_latin_letters_with_diacritics,
    get_class_name,
    log_dataset_statistics,
)
from visualisations.history import plot_history
from visualisations.classification_metrics import get_classification_report

In [7]:
wandb_project = "lowercase_latin_letters_with_diacritics"
train_data = lowercase_latin_letters_with_diacritics(subset="training", path="data/lowercase_latin_letters_with_diacritics/train")
validation_data = lowercase_latin_letters_with_diacritics(subset="validation", path="data/lowercase_latin_letters_with_diacritics/train")
test_data = lowercase_latin_letters_with_diacritics(
  path="data/lowercase_latin_letters_with_diacritics/test",
  subset=None,
  validation_split=None,
)

class_labels = [get_class_name(cn) for cn in validation_data.class_names]

Found 198129 files belonging to 35 classes.
Using 158504 files for training.
Found 198129 files belonging to 35 classes.
Using 39625 files for validation.
Found 22032 files belonging to 35 classes.


In [8]:
logger.info(f"Cleaning wandb project {wandb_project}")
api = wandb.Api()
runs = api.runs(f"gratkadlafana/{wandb_project}")
for run in runs:
    run.delete()
logger.info("Project runs deleted")

2022-11-17 05:51:33.937 | INFO     | __main__:<module>:1 - Cleaning wandb project lowercase_latin_letters_with_diacritics


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 

··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
2022-11-17 06:05:06.884 | INFO     | __main__:<module>:6 - Project runs deleted


In [9]:
with wandb.init(project=wandb_project, config={"class_labels": class_labels}):
  log_dataset_statistics(train_data, validation_data, class_labels)

wandb: Currently logged in as: gratkadlafana. Use `wandb login --relogin` to force relogin


In [ ]:
for model, config in get_models_for_experiment(num_classes=len(class_labels)):
  wandb.init(project=wandb_project, config=config, name=config["model_name"])
  history = train(
      train_data,
      model,
      epochs=50,
      validation_dataset=validation_data,
      callbacks=[
          WandbCallback(),
          tf.keras.callbacks.EarlyStopping(
              monitor="val_loss", patience=4, restore_best_weights=True
          ),
      ],
  )

  _, y_true, y_pred, y_probas = test(test_data, model)
  wandb.sklearn.plot_confusion_matrix(y_true, y_pred)
  wandb.sklearn.plot_roc(y_true, y_probas, class_labels)
  cl = get_classification_report(y_true, y_pred, class_labels=class_labels)
  wandb.log(
      {
          "classification_report": cl,
      }
  )
  wandb.finish()

categorical_accuracy,▁▇▇███████████
epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁
val_categorical_accuracy,▁▆▇▇▇█▇████▇█▇
val_loss,█▃▂▂▁▁▂▁▁▁▁▂▁▁
best_epoch,9
best_val_loss,0.40373
categorical_accuracy,0.79444
epoch,13
loss,0.7178
val_categorical_accuracy,0.86958


Epoch 1/50
4949/4954 [============================>.] - ETA: 0s - loss: 2.3089 - categorical_accuracy: 0.4526

wandb: Adding directory to artifact (/content/wandb/run-20221117_065507-2h95fpvm/files/model-best)... Done. 0.1s


4954/4954 [==============================] - 72s 14ms/step - loss: 2.3078 - categorical_accuracy: 0.4528 - val_loss: 0.6270 - val_categorical_accuracy: 0.8101
Epoch 2/50
4945/4954 [============================>.] - ETA: 0s - loss: 0.8931 - categorical_accuracy: 0.7405

wandb: Adding directory to artifact (/content/wandb/run-20221117_065507-2h95fpvm/files/model-best)... Done. 0.1s


...

Epoch 1/50
4954/4954 [==============================] - ETA: 0s - loss: 46.9699 - categorical_accuracy: 0.6583

wandb: Adding directory to artifact (/content/wandb/run-20221117_090525-33g8rwe0/files/model-best)... Done. 0.3s


4954/4954 [==============================] - 94s 19ms/step - loss: 46.9699 - categorical_accuracy: 0.6583 - val_loss: 3.5564 - val_categorical_accuracy: 0.0269
Epoch 2/50
4952/4954 [============================>.] - ETA: 0s - loss: 775.5604 - categorical_accuracy: 0.0323

wandb: Adding directory to artifact (/content/wandb/run-20221117_090525-33g8rwe0/files/model-best)... Done. 0.2s


4954/4954 [==============================] - 93s 19ms/step - loss: 775.3656 - categorical_accuracy: 0.0323 - val_loss: 3.5537 - val_categorical_accuracy: 0.0322
Epoch 3/50
 844/4954 [====>.........................] - ETA: 1:02 - loss: 3.5540 - categorical_accuracy: 0.0310